In [1]:
%load_ext autoreload
%autoreload 2

import json
from src.db.db_connect import get_db_engine
from src.db.db_schema import get_schema_info

import os
import sys
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# 프로젝트 루트 경로를 시스템 경로에 추가 (src 폴더를 인식하기 위함)
project_root = os.path.abspath(os.path.join(os.getcwd()))
if project_root not in sys.path:
    sys.path.append(project_root)

load_dotenv()
print("✅ 환경 설정 완료")

✅ 환경 설정 완료


In [2]:
import json
from src.db.db_connect import get_db_engine
from src.db.db_schema import get_schema_info

# 1. DB 엔진 연결
engine = get_db_engine()

if engine:
    # 2. 스키마 추출 + LLM 분석(True) + 파일 저장/로드(use_cache=False로 새로 생성)
    # schema_info = get_schema_info(engine, use_llm=True, use_cache=False)
    schema_info = get_schema_info(engine, use_llm=False, use_cache=True)
    
    print("\n" + "="*50)
    print("✅ db_schema.json 파일 생성이 완료되었습니다.")
    print("="*50)
    
    # 3. 결과 확인 (샘플 데이터가 잘 들어갔는지 상단 1개 테이블만 출력)
    first_table = list(schema_info.keys())[0]
    print(f"\n🔍 '{first_table}' 테이블 샘플 데이터 확인:")
    print(json.dumps(schema_info[first_table].get("sample_data", []), indent=2, ensure_ascii=False))

✅ DB 연결 성공: sql_agent
📦 기존에 생성된 스키마 정보(data\db_schema.json)를 활용합니다.

✅ db_schema.json 파일 생성이 완료되었습니다.

🔍 'customers' 테이블 샘플 데이터 확인:
[
  {
    "customer_id": "079bc1808c0fda17593c5642b429bee7",
    "customer_unique_id": "302c1a15f1ada723f22801b30293cde3",
    "customer_zip_code_prefix": 38600,
    "customer_city": "paracatu",
    "customer_state": "MG"
  },
  {
    "customer_id": "0f4e6706036eec942d3b94d1d00ba58f",
    "customer_unique_id": "e3dc9a16e9ff319f01e8be00a9db6c38",
    "customer_zip_code_prefix": 68632,
    "customer_city": "ulianopolis",
    "customer_state": "PA"
  },
  {
    "customer_id": "e507401ad0ed79ab3f14746a0c9a4f9f",
    "customer_unique_id": "19a4d26aef0aadfe7cdc88de373046c6",
    "customer_zip_code_prefix": 8750,
    "customer_city": "mogi das cruzes",
    "customer_state": "SP"
  },
  {
    "customer_id": "48f8c4f0545adf1581cc6c077b7b88f3",
    "customer_unique_id": "fd5233d2c8437ce53379939fa12ad908",
    "customer_zip_code_prefix": 44910,
    "customer_city": "

In [3]:
import os
import json
import datetime
import pandas as pd
import great_expectations as gx
from sqlalchemy import create_engine

# 수정된 클래스 임포트 (Hypothesizer는 Validator 내부에서 쓰므로 생략 가능)
from src.db.integrity_physical import PhysicalIntegrityManager
from src.db.integrity_semantic import SemanticValidator


# 1. 초기화 및 DB 연결
engine = get_db_engine()
context = gx.get_context()
# [STEP 1] 공통 run_id 생성
# 이 ID를 통해 물리/논리 검증 결과가 하나의 실행 단위로 묶입니다.
shared_run_id = f"run_SK_{datetime.datetime.now().strftime('%m%d_%H%M%S')}"
print(f"🆔 통합 실행 ID: {shared_run_id}")

# [STEP 2] Physical 검증 실행
# save_file=False로 설정하여 메모리 상에서 리포트를 관리합니다.
phys_mgr = PhysicalIntegrityManager(engine, context, run_id=shared_run_id)
_, phys_results = phys_mgr.run_validation()
phys_report = phys_mgr.format_results(shared_run_id, phys_results, save_file=False)

# [STEP 3] Semantic 검증 실행
# 조잡한 루프 로직이 SemanticValidator.run_all_semantic_logic 내부로 캡슐화되었습니다.
sem_validator = SemanticValidator(engine, context)
all_sem_results = sem_validator.run_all_semantic_logic(phys_report)
sem_data = sem_validator.get_formatted_report(shared_run_id, all_sem_results)

# [STEP 4] 📂 리포트 병합 (Physical 틀에 Semantic 결과 자연스럽게 주입)
for table, checks in sem_data["tables"].items():
    if table in phys_report["tables"]:
        # Semantic 결과 리스트를 물리 리포트의 checks 뒤에 extend로 추가합니다.
        phys_report["tables"][table]["checks"].extend(checks)
        
        # Semantic 결과 중 FAIL이 있다면 해당 테이블과 전체 요약 상태를 업데이트합니다.
        if any(c.get("status") == "FAIL" for c in checks):
            phys_report["tables"][table]["status"] = "ACTION_REQUIRED"
            phys_report["summary"]["status"] = "ACTION_REQUIRED"

# [STEP 5] 최종 통합 JSON 파일 저장
final_path = "data/db_integrity_result.json"
os.makedirs("data", exist_ok=True)
with open(final_path, "w", encoding="utf-8") as f:
    json.dump(phys_report, f, indent=4, ensure_ascii=False)

print(f"✅ 모든 무결성 검증이 완료되었습니다!")
print(f"📊 최종 통합 리포트: {final_path}")


✅ DB 연결 성공: sql_agent
🆔 통합 실행 ID: run_SK_0325_233718
📦 기존에 생성된 스키마 정보(data\db_schema.json)를 활용합니다.
🚀 [customers] 검증 준비 ->    - 데이터 건수: 99441건
   - PK 목록: ['customer_id']


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

✅ 모든 무결성 검증이 완료되었습니다!
📊 최종 통합 리포트: data/db_integrity_result.json


In [4]:
# 1. 현재 컨텍스트를 기준으로 데이터 문서(대시보드)를 생성합니다.
context.build_data_docs()

# 2. 생성된 대시보드 중 가장 최신 결과 페이지를 브라우저에서 바로 엽니다.
context.open_data_docs()